# Skills gap analysis — starter notebook

This notebook identifies skills gaps across a workforce by comparing employee skill profiles against role competency requirements. It surfaces priority development areas at the individual, team, and organizational level.

**What this notebook does:**
1. Loads employee skill profiles and role competency frameworks
2. Computes gap scores at the individual level
3. Aggregates gaps by team, function, and level
4. Identifies the highest-priority org-wide development needs
5. Generates an L&D investment prioritization output

**Data requirements:**
- Employee skill assessment data (from performance system, skills survey, or LMS)
- Role competency framework (required skills by role and level)
- Employee-to-role mapping from HRIS

**Before using in production:** Complete the [Risk Assessment Template](../03-governance/risk-assessment-template.md). Skills gap data is sensitive — access should be restricted to L&D and HRBPs.

---

In [ ]:
# Requirements
# pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools import product
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Define competency framework

Replace with your organization's competency framework. This defines what skills are required at each role/level combination, and at what proficiency level (1–4).

**Proficiency scale:**
- 1 = Awareness (knows the concept)
- 2 = Developing (can apply with guidance)
- 3 = Proficient (applies independently)
- 4 = Expert (coaches others, sets direction)

In [ ]:
# ── REPLACE WITH YOUR COMPETENCY FRAMEWORK ───────────────────────────────────
# Format: {role: {skill: required_proficiency_level}}

COMPETENCY_FRAMEWORK = {
    'Software Engineer L3': {
        'Python': 3, 'System Design': 2, 'Code Review': 3,
        'Communication': 2, 'Project Management': 1, 'Data Analysis': 2,
        'Security Practices': 2, 'CI/CD': 2, 'Mentoring': 1
    },
    'Software Engineer L4': {
        'Python': 4, 'System Design': 3, 'Code Review': 4,
        'Communication': 3, 'Project Management': 2, 'Data Analysis': 3,
        'Security Practices': 3, 'CI/CD': 3, 'Mentoring': 2
    },
    'HR Business Partner': {
        'Employee Relations': 3, 'Data Analysis': 2, 'Communication': 4,
        'Project Management': 3, 'Change Management': 3, 'Coaching': 3,
        'Employment Law': 2, 'AI Literacy': 2, 'Mentoring': 2
    },
    'Engineering Manager': {
        'Python': 2, 'System Design': 3, 'Communication': 4,
        'Project Management': 4, 'Data Analysis': 3, 'Coaching': 3,
        'Security Practices': 2, 'Mentoring': 4, 'Change Management': 2
    },
    'Recruiter': {
        'Communication': 4, 'Data Analysis': 2, 'Project Management': 2,
        'Employment Law': 2, 'AI Literacy': 2, 'Coaching': 1,
        'Change Management': 1, 'Mentoring': 1, 'Employee Relations': 1
    }
}
# ─────────────────────────────────────────────────────────────────────────────

all_skills = sorted(set(skill for role in COMPETENCY_FRAMEWORK.values() for skill in role))
print(f'Roles in framework: {len(COMPETENCY_FRAMEWORK)}')
print(f'Skills tracked: {len(all_skills)}')
print(f'Skills: {all_skills}')

## 2. Load employee skill profiles

Replace the synthetic data generator with your actual skill assessment export. This could come from:
- A skills assessment survey
- Performance system self-assessments
- LMS completion data mapped to skills
- Manager assessments

In [ ]:
# ── REPLACE WITH YOUR SKILL ASSESSMENT DATA ──────────────────────────────────
# Expected format: one row per employee, columns for each skill (rated 0-4)
# Plus: employee_id, role, department, manager_id, tenure_months

def generate_synthetic_profiles(n=150):
    """Synthetic employee skill profiles for development only."""
    roles = list(COMPETENCY_FRAMEWORK.keys())
    role_weights = [0.35, 0.25, 0.15, 0.15, 0.10]
    departments = {
        'Software Engineer L3': 'Engineering',
        'Software Engineer L4': 'Engineering',
        'HR Business Partner': 'People',
        'Engineering Manager': 'Engineering',
        'Recruiter': 'People'
    }
    records = []
    for i in range(n):
        role = np.random.choice(roles, p=role_weights)
        required = COMPETENCY_FRAMEWORK[role]
        tenure = int(np.random.gamma(3, 12))
        # Skill level correlates loosely with tenure and required level
        skill_levels = {}
        for skill in all_skills:
            req = required.get(skill, 0)
            if req == 0:
                skill_levels[skill] = np.random.choice([0, 1], p=[0.6, 0.4])
            else:
                base = req - np.random.choice([0, 1, 2], p=[0.4, 0.4, 0.2])
                skill_levels[skill] = max(0, min(4, base + (1 if tenure > 24 else 0)))
        records.append({
            'employee_id': f'EMP{i+1:04d}',
            'role': role,
            'department': departments[role],
            'tenure_months': tenure,
            **skill_levels
        })
    return pd.DataFrame(records)

df = generate_synthetic_profiles()
# ─────────────────────────────────────────────────────────────────────────────

print(f'Employees: {len(df):,}')
print(f'Role distribution:')
print(df['role'].value_counts())

## 3. Compute individual gap scores

In [ ]:
def compute_gap(row, skill):
    """Gap = required proficiency - actual proficiency. Negative = exceeds requirement."""
    required = COMPETENCY_FRAMEWORK.get(row['role'], {}).get(skill, 0)
    actual = row.get(skill, 0)
    return max(0, required - actual)  # Only count deficits, not excesses

# Compute gap for each skill
gap_cols = []
for skill in all_skills:
    col = f'gap_{skill}'
    df[col] = df.apply(lambda row: compute_gap(row, skill), axis=1)
    gap_cols.append(col)

# Overall gap score per employee (sum of all gaps)
df['total_gap_score'] = df[gap_cols].sum(axis=1)
df['skills_at_gap'] = (df[gap_cols] > 0).sum(axis=1)

print('Gap score distribution:')
print(df['total_gap_score'].describe().round(2))
print()
print('Employees with zero gaps:', (df['total_gap_score'] == 0).sum())
print('Employees with 3+ skill gaps:', (df['skills_at_gap'] >= 3).sum())

## 4. Org-wide priority gaps

In [ ]:
# Which skills have the most widespread gaps across the org?
gap_summary = pd.DataFrame({
    'skill': all_skills,
    'employees_with_gap': [( df[f'gap_{s}'] > 0).sum() for s in all_skills],
    'avg_gap_size': [df[f'gap_{s}'].mean().round(2) for s in all_skills],
    'pct_workforce_affected': [(df[f'gap_{s}'] > 0).mean().round(3) * 100 for s in all_skills]
}).sort_values('employees_with_gap', ascending=False)

print('Top 10 org-wide skill gaps:')
print(gap_summary.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
top_gaps = gap_summary.head(8)
bars = ax.barh(top_gaps['skill'], top_gaps['pct_workforce_affected'],
               color='#185FA5', alpha=0.85)
ax.set_xlabel('% of workforce with a gap in this skill')
ax.set_title('Priority skills gaps — % of workforce affected', fontsize=13)
ax.invert_yaxis()
for bar, val in zip(bars, top_gaps['pct_workforce_affected']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('skills_gap_priority.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Gap heatmap by role

In [ ]:
# Average gap by role and skill — shows where each team needs the most development
role_skill_gaps = df.groupby('role')[[f'gap_{s}' for s in all_skills]].mean()
role_skill_gaps.columns = all_skills

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    role_skill_gaps,
    annot=True, fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Avg gap (0 = no gap, 3 = large gap)'}
)
ax.set_title('Average skill gap by role', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('skills_gap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. L&D investment prioritization

In [ ]:
# Score each skill for L&D investment priority:
# High reach (many affected) + high severity (large average gap) = highest priority

gap_summary['reach_score'] = (gap_summary['pct_workforce_affected'] /
                               gap_summary['pct_workforce_affected'].max())
gap_summary['severity_score'] = (gap_summary['avg_gap_size'] /
                                  gap_summary['avg_gap_size'].max())
gap_summary['priority_score'] = (
    0.6 * gap_summary['reach_score'] +
    0.4 * gap_summary['severity_score']
).round(3)

gap_summary['priority_tier'] = pd.cut(
    gap_summary['priority_score'],
    bins=[0, 0.33, 0.66, 1.01],
    labels=['Tier 3 — monitor', 'Tier 2 — plan', 'Tier 1 — invest now']
)

priority_output = gap_summary[[
    'skill', 'employees_with_gap', 'pct_workforce_affected',
    'avg_gap_size', 'priority_score', 'priority_tier'
]].sort_values('priority_score', ascending=False)

print('L&D investment prioritization:')
print(priority_output.to_string(index=False))

priority_output.to_csv('ld_investment_priorities.csv', index=False)
print('\nExported to: ld_investment_priorities.csv')

## Next steps

Before using this analysis to drive L&D investment decisions:

1. **Validate data quality** — skill self-assessments are notoriously biased (Dunning-Kruger in both directions). Consider calibrating with manager assessments or objective measures where possible.

2. **Weight by strategic importance** — not all skills are equally important to the business right now. Layer in a strategic weight before finalizing priorities (e.g., AI Literacy may be a strategic priority even if its gap score is moderate).

3. **Check for equity** — run gap scores by demographic group before publishing. If one group consistently shows higher gaps, the issue may be access to development opportunities, not skill level.

4. **Build the business case** — pair gap data with the cost of the gap (lost productivity, attrition, project failures) to make the ROI argument for L&D investment.

5. **Share with HRBPs** — individual-level gap data should only go to HRBPs for their teams, not to managers directly without HR review.